In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/ship_fuel_efficiency.csv')

## Initial Data Inspection

Let's take a look at the first few rows of the dataset to understand its structure.

In [2]:
display(df.head())

,ship_id,ship_type,route_id,month,distance,fuel_type,fuel_consumption,CO2_emissions,weather_conditions,engine_efficiency
0,NG001,Oil Service Boat,Warri-Bonny,January,132.26,HFO,3779.77,10625.76,Stormy,92.14
1,NG001,Oil Service Boat,Port Harcourt-Lagos,February,128.52,HFO,4461.44,12779.73,Moderate,92.98
2,NG001,Oil Service Boat,Port Harcourt-Lagos,March,67.30,HFO,1867.73,5353.01,Calm,87.61
3,NG001,Oil Service Boat,Port Harcourt-Lagos,April,71.68,Diesel,2393.51,6506.52,Stormy,87.42
4,NG001,Oil Service Boat,Lagos-Apapa,May,134.32,HFO,4267.19,11617.03,Calm,85.61


Next, let's examine the data types and check for any missing values using `info()` and `describe()`.

In [3]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ship_id             1440 non-null   object 
 1   ship_type           1440 non-null   object 
 2   route_id            1440 non-null   object 
 3   month               1440 non-null   object 
 4   distance            1440 non-null   float64
 5   fuel_type           1440 non-null   object 
 6   fuel_consumption    1440 non-null   float64
 7   CO2_emissions       1440 non-null   float64
 8   weather_conditions  1440 non-null   object 
 9   engine_efficiency   1440 non-null   float64
dtypes: float64(4), object(6)
memory usage: 112.6+ KB
None
          distance  fuel_consumption  CO2_emissions  engine_efficiency
count  1440.000000       1440.000000    1440.000000        1440.000000
mean    151.753354       4844.246535   13365.454882          82.582924
std     108.472230       4892.352

### Checking for Duplicates

Let's check if there are any duplicate rows in the dataset and remove them if found.

In [4]:
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
duplicates_removed = initial_rows - df.shape[0]

print(f"Number of duplicate rows removed: {duplicates_removed}")
print(f"Number of rows after removing duplicates: {df.shape[0]}")

Number of duplicate rows removed: 0
Number of rows after removing duplicates: 1440


### Data Type Conversion for 'month'

The `month` column is currently an `object` type. Converting it to an ordered categorical type will ensure proper chronological sorting in any visualizations or filters.

In [5]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)

print("Data types after converting 'month' to ordered categorical:")
print(df.info())

Data types after converting 'month' to ordered categorical:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1440 entries, 0 to 1439
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   ship_id             1440 non-null   object  
 1   ship_type           1440 non-null   object  
 2   route_id            1440 non-null   object  
 3   month               1440 non-null   category
 4   distance            1440 non-null   float64 
 5   fuel_type           1440 non-null   object  
 6   fuel_consumption    1440 non-null   float64 
 7   CO2_emissions       1440 non-null   float64 
 8   weather_conditions  1440 non-null   object  
 9   engine_efficiency   1440 non-null   float64 
dtypes: category(1), float64(4), object(5)
memory usage: 103.2+ KB
None


## KPI Calculation and Data Preparation for Dashboard

Based on the `ship_fuel_efficiency.csv` dataset, we will calculate the following KPIs:
- **Total Fuel Consumption**
- **Total CO2 Emissions**
- **Average Engine Efficiency**
- **Top 5 Ships by Fuel Consumption**
- **Top 5 Routes by CO2 Emissions**

We will also prepare aggregated data for the Streamlit dashboard, incorporating filters for `ship_type`, `route_id`, and `fuel_type`.

In [6]:
# 1. Total Fuel Consumption
total_fuel_consumption = df['fuel_consumption'].sum()

# 2. Total CO2 Emissions
total_co2_emissions = df['CO2_emissions'].sum()

# 3. Average Engine Efficiency
average_engine_efficiency = df['engine_efficiency'].mean()

# 4. Top 5 Ships by Fuel Consumption
top_5_ships_fuel = df.groupby('ship_id')['fuel_consumption'].sum().nlargest(5).reset_index()

# 5. Top 5 Routes by CO2 Emissions
top_5_routes_co2 = df.groupby('route_id')['CO2_emissions'].sum().nlargest(5).reset_index()

print(f"Total Fuel Consumption: {total_fuel_consumption:.2f}")
print(f"Total CO2 Emissions: {total_co2_emissions:.2f}")
print(f"Average Engine Efficiency: {average_engine_efficiency:.2f}")

print("\nTop 5 Ships by Fuel Consumption:")
display(top_5_ships_fuel)

print("\nTop 5 Routes by CO2 Emissions:")
display(top_5_routes_co2)

Total Fuel Consumption: 6975715.01
Total CO2 Emissions: 19246255.03
Average Engine Efficiency: 82.58

Top 5 Ships by Fuel Consumption:


,ship_id,fuel_consumption
0,NG012,163730.38
1,NG071,160758.76
2,NG037,156795.82
3,NG048,154938.39
4,NG016,152941.71



Top 5 Routes by CO2 Emissions:


,route_id,CO2_emissions
0,Lagos-Apapa,5516231.71
1,Port Harcourt-Lagos,5135239.40
2,Escravos-Lagos,5045208.21
3,Warri-Bonny,3549575.71


### Data Aggregation for Streamlit Dashboard

To support the dashboard filters, we will create aggregated datasets. We'll group by `ship_type`, `route_id`, `fuel_type`, and `month` to allow for interactive filtering and time-series analysis.

In [7]:
grouped_data = df.groupby(['ship_type', 'route_id', 'fuel_type', 'month']).agg(
    total_fuel_consumption=('fuel_consumption', 'sum'),
    total_co2_emissions=('CO2_emissions', 'sum'),
    average_engine_efficiency=('engine_efficiency', 'mean')
).reset_index()

print("Aggregated Data for Dashboard:")
display(grouped_data.head())

Aggregated Data for Dashboard:


/tmp/ipykernel_8571/3412951566.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_data = df.groupby(['ship_type', 'route_id', 'fuel_type', 'month']).agg(


,ship_type,route_id,fuel_type,month,total_fuel_consumption,total_co2_emissions,average_engine_efficiency
0,Fishing Trawler,Escravos-Lagos,Diesel,January,20742.60,57699.13,80.267143
1,Fishing Trawler,Escravos-Lagos,Diesel,February,5230.98,14643.04,81.855000
2,Fishing Trawler,Escravos-Lagos,Diesel,March,9709.13,26366.07,77.887500
3,Fishing Trawler,Escravos-Lagos,Diesel,April,5769.06,15440.18,78.915000
4,Fishing Trawler,Escravos-Lagos,Diesel,May,7583.57,20634.75,85.130000


## Build a Streamlit Dashboard

Now, let's create the Streamlit dashboard with the calculated KPIs and interactive filters. This dashboard will allow users to explore the data based on `ship_type`, `route_id`, and `fuel_type`.

**Note:** To run this Streamlit app, you will need to save the code below as a Python file (e.g., `app.py`) and then execute it from your terminal using `streamlit run app.py`.

First, let's ensure we have the necessary libraries installed:

In [8]:
# Install Streamlit and Altair if not already installed
!pip install streamlit altair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 55.7 MB/s eta 0:00:00


### Streamlit Dashboard Code (`app.py`)

This code will set up the Streamlit interface, display the KPIs, and provide interactive filters and charts.

In [9]:
%%writefile app.py
import streamlit as st
import pandas as pd
import altair as alt

# Load the dataset (assuming it's available in the same directory or path)
df = pd.read_csv('/content/ship_fuel_efficiency.csv')

# Data Cleaning and Preparation (as performed previously)
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)

# Calculate KPIs
total_fuel_consumption = df['fuel_consumption'].sum()
total_co2_emissions = df['CO2_emissions'].sum()
average_engine_efficiency = df['engine_efficiency'].mean()

# Top 5 Ships by Fuel Consumption
top_5_ships_fuel = df.groupby('ship_id')['fuel_consumption'].sum().nlargest(5).reset_index()

# Top 5 Routes by CO2 Emissions
top_5_routes_co2 = df.groupby('route_id')['CO2_emissions'].sum().nlargest(5).reset_index()

# Streamlit Dashboard Layout
st.set_page_config(layout="wide")
st.title("Ship Fuel Efficiency & Emissions Dashboard")

# Sidebar for Filters
st.sidebar.header("Filter Options")

# Dynamic Filters
selected_ship_type = st.sidebar.multiselect(
    "Select Ship Type",
    options=df['ship_type'].unique(),
    default=df['ship_type'].unique()
)

selected_route_id = st.sidebar.multiselect(
    "Select Route ID",
    options=df['route_id'].unique(),
    default=df['route_id'].unique()
)

selected_fuel_type = st.sidebar.multiselect(
    "Select Fuel Type",
    options=df['fuel_type'].unique(),
    default=df['fuel_type'].unique()
)

# Apply filters
filtered_df = df[
    df['ship_type'].isin(selected_ship_type) &
    df['route_id'].isin(selected_route_id) &
    df['fuel_type'].isin(selected_fuel_type)
]

# Display KPIs
st.subheader("Key Performance Indicators")
col1, col2, col3 = st.columns(3)
col1.metric("Total Fuel Consumption", f"{filtered_df['fuel_consumption'].sum():,.2f} Liters")
col2.metric("Total CO2 Emissions", f"{filtered_df['CO2_emissions'].sum():,.2f} kg")
col3.metric("Average Engine Efficiency", f"{filtered_df['engine_efficiency'].mean():,.2f} %")

# Charts for KPIs
st.subheader("Detailed KPI Analysis")

# Group data for charts
chart_data = filtered_df.groupby('month').agg(
    total_fuel_consumption=('fuel_consumption', 'sum'),
    total_co2_emissions=('CO2_emissions', 'sum'),
    average_engine_efficiency=('engine_efficiency', 'mean')
).reset_index()

# Chart 1: Monthly Fuel Consumption
chart_fuel = alt.Chart(chart_data).mark_line(point=True).encode(
    x=alt.X('month', sort=month_order, title='Month'),
    y=alt.Y('total_fuel_consumption', title='Total Fuel Consumption (Liters)'),
    tooltip=['month', 'total_fuel_consumption']
).properties(title='Monthly Total Fuel Consumption')
st.altair_chart(chart_fuel, use_container_width=True)

# Chart 2: Monthly CO2 Emissions
chart_co2 = alt.Chart(chart_data).mark_line(point=True).encode(
    x=alt.X('month', sort=month_order, title='Month'),
    y=alt.Y('total_co2_emissions', title='Total CO2 Emissions (kg)'),
    tooltip=['month', 'total_co2_emissions']
).properties(title='Monthly Total CO2 Emissions')
st.altair_chart(chart_co2, use_container_width=True)

# Chart 3: Monthly Average Engine Efficiency
chart_efficiency = alt.Chart(chart_data).mark_line(point=True).encode(
    x=alt.X('month', sort=month_order, title='Month'),
    y=alt.Y('average_engine_efficiency', title='Average Engine Efficiency (%)'),
    tooltip=['month', 'average_engine_efficiency']
).properties(title='Monthly Average Engine Efficiency')
st.altair_chart(chart_efficiency, use_container_width=True)

# Top N Charts
st.subheader("Top Performers")
col4, col5 = st.columns(2)

# Chart 4: Top 5 Ships by Fuel Consumption
top_5_ships_fuel_filtered = filtered_df.groupby('ship_id')['fuel_consumption'].sum().nlargest(5).reset_index()
chart_top_ships = alt.Chart(top_5_ships_fuel_filtered).mark_bar().encode(
    x=alt.X('fuel_consumption', title='Total Fuel Consumption (Liters)'),
    y=alt.Y('ship_id', sort='-x', title='Ship ID'),
    tooltip=['ship_id', 'fuel_consumption']
).properties(title='Top 5 Ships by Fuel Consumption')
col4.altair_chart(chart_top_ships, use_container_width=True)

# Chart 5: Top 5 Routes by CO2 Emissions
top_5_routes_co2_filtered = filtered_df.groupby('route_id')['CO2_emissions'].sum().nlargest(5).reset_index()
chart_top_routes = alt.Chart(top_5_routes_co2_filtered).mark_bar().encode(
    x=alt.X('CO2_emissions', title='Total CO2 Emissions (kg)'),
    y=alt.Y('route_id', sort='-x', title='Route ID'),
    tooltip=['route_id', 'CO2_emissions']
).properties(title='Top 5 Routes by CO2 Emissions')
col5.altair_chart(chart_top_routes, use_container_width=True)

# Instructions to run the app
st.sidebar.markdown("""
---
**How to Run:**
1. Save this code as `app.py`
2. Open your terminal.
3. Navigate to the directory where you saved `app.py`.
4. Run `streamlit run app.py`
""")

Writing app.py
